In [ ]:
# Run this in a cell or a quick script to sanity-check
import tflite_runtime.interpreter as tflite
import numpy as np
from PIL import Image

interpreter = tflite.Interpreter(model_path="models/tflite/efficientnet_b0_int8.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])   # Should be [1, 224, 224, 3]
print("Input dtype:", input_details[0]['dtype'])   # float32
print("Output shape:", output_details[0]['shape']) # Should be [1, 38]

# Feed a test image
img = Image.open("data/raw/PlantVillage/Tomato___Late_blight/0a5e9323-dbad-432d-ac58-d291718345d9___RS_LB 4494.jpg").convert("RGB")
img = img.resize((224, 224))
arr = np.array(img, dtype=np.float32) / 255.0
arr = (arr - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
arr = np.expand_dims(arr, 0)

interpreter.set_tensor(input_details[0]['index'], arr)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

top5 = np.argsort(output[0])[::-1][:5]
import json
with open("models/class_names.json") as f:
    class_names = json.load(f)

for i in top5:
    print(f"  {class_names[i]}: {output[0][i]:.4f}")

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import random

DATA_DIR = Path("data/raw/PlantVillage")

classes = sorted(os.listdir(DATA_DIR))
print(f"Total classes: {len(classes)}")

# Count images per class
counts = {cls: len(os.listdir(DATA_DIR / cls)) for cls in classes}
print(f"Total images: {sum(counts.values())}")

# Show 9 random samples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax in axes.flat:
    cls = random.choice(classes)
    img_path = random.choice(list((DATA_DIR / cls).iterdir()))
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(cls.replace("___", "\n"), fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_images.png")
plt.show()